# Memory Is All You Need - Comparison with Standard Transformer

This notebook demonstrates the advantages of the **MemNet** architecture (with active long-term memory) over a standard transformer baseline on the **Copy Task** with a long delay.

Task: The model sees a short random sequence of tokens, a delimiter, a padding area, followed by a long delay of zeros, and must reproduce the original sequence after the delay.

A standard transformer struggles with very long dependencies without memory mechanisms. MemNet retains information through active consolidation.

In [ ]:
#import os
#if not os.path.exists('memory-is-all-you-need'):
#    !git clone https://github.com/Ant1pozitive/memory-is-all-you-need.git
#
#%cd memory-is-all-you-need
#!pip install -r requirements.txt
#
#from IPython.display import clear_output
#clear_output()

In [ ]:
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import DataLoader, random_split

# Original imports
from data.copy_dataset import CopyDataset
from config import cfg
from model.memnet import MemNet
from train import train_epoch, evaluate, sparsity_loss, utilization_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Baseline Transformer (full-sequence attention)

In [ ]:
class BaselineTransformer(nn.Module):
    def __init__(self, vocab_size=20, embed_dim=128, num_layers=4, num_heads=8, hidden_dim=512, max_len=1024):
        super().__init__()
        self.embed = nn.Embedding(vocab_size + 1, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_len, embed_dim))
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads,
                                           dim_feedforward=hidden_dim, batch_first=True,
                                           activation='gelu', norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        emb = self.embed(x) + self.pos_embed[:, :T, :]
        out = self.transformer(emb)
        return self.head(out)

## 2. Dataset & Settings

In [ ]:
cfg.task.seq_len = 10
cfg.task.delay_len = 100  # Increase to 300-500 to show larger gap
cfg.train.batch_size = 8  # This is safe for my poor 8 Gb GPU; You can increase if you can
cfg.train.mixed_precision = True

full_dataset = CopyDataset(
    vocab_size=cfg.model.vocab_size,
    seq_len=cfg.task.seq_len,
    delay_len=cfg.task.delay_len,
    size=5000
)

train_size = int(0.8 * len(full_dataset))
train_dataset, val_dataset = random_split(full_dataset, [train_size, len(full_dataset) - train_size])

train_loader = DataLoader(train_dataset, batch_size=cfg.train.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.train.batch_size, shuffle=False)

print(f"Sequence length: ~{cfg.task.seq_len * 2 + cfg.task.delay_len + 1}")
print(f"Delay: {cfg.task.delay_len}, Batch size: {cfg.train.batch_size}")

## 3. Train Baseline

In [ ]:
baseline = BaselineTransformer(vocab_size=cfg.model.vocab_size).to(device)
optimizer_baseline = torch.optim.AdamW(baseline.parameters(), lr=cfg.train.lr)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

baseline_accs = []
for epoch in tqdm(range(1, 26), desc="Training Baseline"):
    baseline.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = baseline(x)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        optimizer_baseline.zero_grad()
        loss.backward()
        optimizer_baseline.step()
    
    # Validation
    baseline.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = baseline(x)
            pred = logits.argmax(-1)
            mask = y != -100
            correct += (pred[mask] == y[mask]).sum().item()
            total += mask.sum().item()
    acc = correct / total if total > 0 else 0
    baseline_accs.append(acc)

print(f"Best Baseline Accuracy: {max(baseline_accs):.4f}")

## 4. Train MemNet

In [ ]:
memnet = MemNet(cfg).to(device)
optimizer = torch.optim.AdamW(memnet.parameters(), lr=cfg.train.lr)
scaler = GradScaler(enabled=cfg.train.mixed_precision)

memnet_accs = []
for epoch in tqdm(range(1, 26), desc="Training MemNet"):
    train_epoch(memnet, train_loader, optimizer, scaler, epoch)
    _, val_acc = evaluate(memnet, val_loader, epoch)
    memnet_accs.append(val_acc)

print(f"Best MemNet Accuracy: {max(memnet_accs):.4f}")

## 5. Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_memory_dynamics(model, dataset):
    model.eval()
    # Get a single sample
    inputs, targets = next(iter(torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True)))
    inputs = inputs.to(cfg.device)
    
    with torch.no_grad():
        # Get full returns including attention weights
        logits, recon, (w_stm, w_ltm), w_write = model(inputs, return_attn=True)
    
    # Weights shape: [Batch, Heads, Time_Step, Slots] -> Take mean over heads
    # Resulting shape for plot: [Time_Step, Slots]
    # We transpose to [Slots, Time_Step] for heatmap
    
    stm_map = w_stm[0].mean(dim=0).cpu().numpy().T
    ltm_map = w_ltm[0].mean(dim=0).cpu().numpy().T
    
    # Focus only on the 'recall' phase (last N steps) to see retrieval
    seq_len = inputs.shape[1]
    
    plt.figure(figsize=(20, 10))
    
    # 1. STM Activation Map
    plt.subplot(2, 1, 1)
    sns.heatmap(stm_map, cmap="viridis", vmin=0, vmax=0.1)
    plt.title(f"Short-Term Memory (STM) Activation | Entropy: High (Recent)", fontsize=14)
    plt.ylabel("Memory Slots")
    plt.xlabel("Time Step (Sequence)")

    # 2. LTM Activation Map
    plt.subplot(2, 1, 2)
    sns.heatmap(ltm_map, cmap="magma", vmin=0, vmax=0.1)
    plt.title(f"Long-Term Memory (LTM) Activation | Retrieval via Mahalanobis Metric", fontsize=14)
    plt.ylabel("Memory Slots")
    plt.xlabel("Time Step (Sequence)")
    
    plt.tight_layout()
    plt.show()

    # 3. Decision Confidence (Cognitive Load)
    # Plot the max attention weight over time to see "certainty"
    stm_conf = w_stm[0].mean(dim=0).max(dim=1).values.cpu().numpy()
    ltm_conf = w_ltm[0].mean(dim=0).max(dim=1).values.cpu().numpy()
    
    plt.figure(figsize=(12, 4))
    plt.plot(stm_conf, label="STM Confidence (System 1)", color='cyan', alpha=0.8)
    plt.plot(ltm_conf, label="LTM Confidence (System 2)", color='purple', alpha=0.8)
    plt.title("System 1 vs System 2: Confidence Dynamics over Time")
    plt.xlabel("Time Step")
    plt.ylabel("Max Attention Weight")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

print("Visualizing Cognitive Internal State...")
visualize_memory_dynamics(memnet, val_dataset)

## Conclusion

- The baseline transformer plateaus at low accuracy for long delays due to attention dilution.
- **MemNet** achieves significantly higher accuracy by actively consolidating information in memory.

Experiment with even longer `cfg.task.delay_len` (e.g., 500+) to see the memory advantage grow... GLHF!